# Phase C — 10 Attention Mechanisms Investigation (E06–E15) on Winning PDN-S Backbone

Evaluates **10 attention mechanisms** inserted at the exact same point in the Phase B winning **`PDN-S` Student** (`conv3` output: `(B, 256, 59, 59)` → `(B, 256, 59, 59)`), under the exact same controlled protocol as `E01_PUAD_200ep` (`89.64%` Combined Mean / `84.07%` Pooled Raw):

- **E06**: SE (Squeeze-and-Excitation)
- **E07**: ECA (Efficient Channel Attention)
- **E08**: CBAM (Convolutional Block Attention Module)
- **E09**: Coordinate Attention (CoordAtt)
- **E10**: Triplet Attention
- **E11**: SimAM (Parameter-Free 3D Energy)
- **E12**: GCT (Gated Channel Transformation)
- **E13**: BAM (Bottleneck Attention Module)
- **E14**: Non-Local Self-Attention
- **E15**: Criss-Cross Attention (R=2)

All runs auto-save every epoch to Google Drive and auto-skip if already completed, so you can run all 10 across 1 or multiple Colab sessions effortlessly.

In [ ]:
#@title 1. Section 3 — Hardware, Runtime, RAM & Disk Audit
import os, shutil, sys, torch, psutil
print('=== COLAB SESSION AUDIT (PHASE C) ===')
print('Python        :', sys.version.split()[0])
print('PyTorch       :', torch.__version__, '| CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'No GPU assigned!'
props = torch.cuda.get_device_properties(0)
print(f'GPU Model     : {props.name} (CC {props.major}.{props.minor}) | VRAM: {props.total_memory/(1024**3):.2f} GiB | Native BF16: {props.major >= 8 and torch.cuda.is_bf16_supported()}')
vm = psutil.virtual_memory()
print(f'System RAM    : {vm.available/(1024**3):.1f} GB free / {vm.total/(1024**3):.1f} GB total')
disk = shutil.disk_usage('/content')
print(f'Local SSD     : {disk.free/(1024**3):.1f} GB free / {disk.total/(1024**3):.1f} GB total')

In [ ]:
#@title 2. Mount Drive, Pull Latest Code & Ensure Local SSD Datasets
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_LOCO_TAR    = f'{DRIVE_PROJECT_DIR}/data/mvtec_loco_anomaly_detection.tar.xz'
DRIVE_IMNET_ZIP   = f'{DRIVE_PROJECT_DIR}/data/imagenet_penalty_10k.zip'
DRIVE_TEACHER_PT  = f'{DRIVE_PROJECT_DIR}/weights/teacher_pdn_s.pt'
DRIVE_RESULTS_DIR = f'{DRIVE_PROJECT_DIR}/runs'
GITHUB_REPO_URL   = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'

REPO_DIR = '/content/Endsem'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)

!pip install -q timm thop einops pyyaml psutil tqdm
%cd /content/Endsem

LOCAL_LOCO = '/content/data/mvtec_loco_anomaly_detection'
LOCAL_IMNET = '/content/data/imagenet_penalty'
os.makedirs('/content/data', exist_ok=True)
if not os.path.isdir(os.path.join(LOCAL_LOCO, 'breakfast_box')):
    os.makedirs(LOCAL_LOCO, exist_ok=True)
    !tar -xf "$DRIVE_LOCO_TAR" -C "$LOCAL_LOCO"
if not os.path.isdir(LOCAL_IMNET):
    !unzip -q "$DRIVE_IMNET_ZIP" -d /content/data/

os.environ['ENDSEM_LOCO_ROOT'] = LOCAL_LOCO
os.environ['ENDSEM_IMAGENET_ROOT'] = LOCAL_IMNET
os.environ['ENDSEM_TEACHER_PATH'] = DRIVE_TEACHER_PT
!python scripts/verify_env.py

In [ ]:
#@title 3. Batch 1 — Channel & Hybrid Attentions (E06 SE, E07 ECA, E08 CBAM, E09 CoordAtt, E10 Triplet)
!python scripts/run_experiment.py --config configs/E06_se.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E07_eca.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E08_cbam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E09_coordatt.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E10_triplet.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 4. Batch 2 — Energy, Gated, Dilated & Global Attentions (E11 SimAM, E12 GCT, E13 BAM, E14 NonLocal, E15 CrissCross)
!python scripts/run_experiment.py --config configs/E11_simam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E12_gct.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E13_bam.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E14_nonlocal.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E15_crisscross.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

In [ ]:
#@title 5. Aggregate & Compare All Experiments (E01–E15 Master Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseC_summary"